The purpose of this notebook is to download and clean the election data for the last two Danish general elections (2019 and 2022).

These results (at the municipality level) are contained in the ``FVKOM`` table in the Danish Statbank

In [65]:
import requests
import pandas as pd
from IPython.display import display
from io import StringIO
from dstapi import DstApi 

In [66]:
election_api = DstApi('FVKOM')
election_api.tablesummary(language='en')


Table FVKOM: Election to the Parliament by result of the election, region and time
Last update: 2022-11-18T08:00:00


,variable name,# values,First value,First value label,Last value,Last value label,Time variable
0,VALRES,50,GYLD_IALT,VALID VOTES,VAELG,NUMBER OF VOTERS,False
1,OMRÅDE,99,101,Copenhagen,851,Aalborg,False
2,Tid,5,2007,2007,2022,2022,True


In [67]:
params = election_api.define_base_params(language='en')

df = election_api.get_data(params=params)
df.head()

,VALRES,OMRÅDE,TID,INDHOLD
0,ABSENTEE BALLOTS CONSIDERED (RECEIVED BEFORE D...,Copenhagen,2019,39158
1,ABSENTEE BALLOTS CONSIDERED (RECEIVED BEFORE D...,Frederiksberg,2019,9208
2,ABSENTEE BALLOTS CONSIDERED (RECEIVED BEFORE D...,Ballerup,2019,2942
3,ABSENTEE BALLOTS CONSIDERED (RECEIVED BEFORE D...,Brøndby,2019,2014
4,ABSENTEE BALLOTS CONSIDERED (RECEIVED BEFORE D...,Dragør,2019,878


In [68]:
# Translate column names
df.rename(columns={
    'VALRES': 'party',
    'OMRÅDE': 'municipality',
    'TID': 'year',
    'INDHOLD': 'N'
}, inplace=True)

# Keep only 2019 and 2022
df = df[df['year'].isin([2019, 2022])]

print(f"Shape: {df.shape}")
print(f"Years: {df['year'].unique()}")
df.head()

Shape: (7548, 4)
Years: [2019 2022]


,party,municipality,year,N
0,ABSENTEE BALLOTS CONSIDERED (RECEIVED BEFORE D...,Copenhagen,2019,39158
1,ABSENTEE BALLOTS CONSIDERED (RECEIVED BEFORE D...,Frederiksberg,2019,9208
2,ABSENTEE BALLOTS CONSIDERED (RECEIVED BEFORE D...,Ballerup,2019,2942
3,ABSENTEE BALLOTS CONSIDERED (RECEIVED BEFORE D...,Brøndby,2019,2014
4,ABSENTEE BALLOTS CONSIDERED (RECEIVED BEFORE D...,Dragør,2019,878


In [69]:
print(df['party'].unique())

['ABSENTEE BALLOTS CONSIDERED (RECEIVED BEFORE DEADLINE)' 'VALID VOTES'
 'PERSONAL VOTES' 'Independent candidates'
 'INVALID VOTES CAST ON POLLING STATION' 'INVALID ADVANCE VOTES'
 'INVALID VOTES' 'NUMBER OF VOTERS' 'Blanks'
 'Ballot not provided by the polling station'
 'Other mark than cross used in addition to cross'
 'More than three crosses' 'Ballot with drawing, writing or sticker'
 'Ballot marked with different colours, torn, or other characteristic'
 'Å. The Alternative' 'D. The New Right' 'E. Klaus Riskær Pedersen'
 'P. Hard Line' 'Other mark than cross used'
 'Only name for person not electable or uncertain unofficial letter '
 'Only unofficial letter (one letter)'
 'Only uncertain unofficial letter (more than one letter)' 'Only cross'
 'Inconsistency between letter, party or person'
 'Other reasons to doubt the voter´s intention'
 'Envelope contained more than one ballot or other material'
 'Ballot not provided by the minister' 'Partly or completely crossed over'
 "Q. Indepe

In [70]:
party_remap = {
    'A. The Social Democratic Party': 'A: Socialdemokratiet',
    'B. The Danish Social-Liberal Party': 'B: Det Radikale Venstre',
    'C. The Conservative Peoples Party': 'C: Det Konservative Folkeparti',
    'D. The New Right': 'D: Nye Borgerlige',
    'E. Klaus Riskær Pedersen': 'Another party',
    'F. The Socialist Peoples Party': 'F: SF - Socialistisk Folkeparti',
    'I. Liberal Alliance': 'I: Liberal Alliance',
    'K. Christian Democrats': 'K: Kristendemokraterne',
    'M. The Moderats': 'M: Moderaterne',
    'O. The Danish Peoples Party': 'O: Dansk Folkeparti',
    'P. Hard Line': 'Another party',
    'Q. Independent Greens - Denmark\'s New Left-Wing Party': 'Q: Frie Grønne',
    'V. Venstre, Denmarks Liberal Party': 'V: Venstre',
    'Æ. The Danish Democrats - Inger Støjberg': 'Æ: Danmarksdemokraterne',
    'Å. The Alternative': 'Å: Alternativet',
    'Ø. The Red-Green Alliance': 'Ø: Enhedslisten',
    'Independent candidates': 'Independent candidate',
}

# Keep only party/independent rows, then remap
df = df[df['party'].isin(party_remap.keys())].copy()
df['party'] = df['party'].map(party_remap)

print(f"Shape: {df.shape}")
print(df['party'].unique())

Shape: (2831, 4)
['Independent candidate' 'Å: Alternativet' 'D: Nye Borgerlige'
 'Another party' 'Q: Frie Grønne' 'M: Moderaterne'
 'Æ: Danmarksdemokraterne' 'A: Socialdemokratiet'
 'B: Det Radikale Venstre' 'C: Det Konservative Folkeparti'
 'F: SF - Socialistisk Folkeparti' 'O: Dansk Folkeparti'
 'K: Kristendemokraterne' 'V: Venstre' 'Ø: Enhedslisten'
 'I: Liberal Alliance']


In [71]:
# Calculate vote percentage per party within each municipality and year
df['N_pct'] = df['N'] / df.groupby(['municipality', 'year'])['N'].transform('sum')
df.head(10)

,party,municipality,year,N,N_pct
297,Independent candidate,Copenhagen,2019,506,0.001432
298,Independent candidate,Frederiksberg,2019,70,0.001078
299,Independent candidate,Ballerup,2019,82,0.002839
300,Independent candidate,Brøndby,2019,69,0.003751
301,Independent candidate,Dragør,2019,5,0.000552
302,Independent candidate,Gentofte,2019,106,0.002367
303,Independent candidate,Gladsaxe,2019,93,0.002389
304,Independent candidate,Glostrup,2019,39,0.002987
305,Independent candidate,Herlev,2019,59,0.003558
306,Independent candidate,Albertslund,2019,44,0.002924


In [72]:
sample = df[(df['municipality'] == 'Copenhagen') & (df['year'] == 2022)].sort_values('N_pct', ascending=False)
print(sample.to_string(index=False))

                          party municipality  year     N    N_pct
           A: Socialdemokratiet   Copenhagen  2022 63042 0.177002
                Ø: Enhedslisten   Copenhagen  2022 54246 0.152306
F: SF - Socialistisk Folkeparti   Copenhagen  2022 42762 0.120062
                Å: Alternativet   Copenhagen  2022 36877 0.103539
                 M: Moderaterne   Copenhagen  2022 32677 0.091747
            I: Liberal Alliance   Copenhagen  2022 29623 0.083172
        B: Det Radikale Venstre   Copenhagen  2022 27910 0.078363
                     V: Venstre   Copenhagen  2022 27867 0.078242
 C: Det Konservative Folkeparti   Copenhagen  2022 15173 0.042601
                 Q: Frie Grønne   Copenhagen  2022  8852 0.024854
        Æ: Danmarksdemokraterne   Copenhagen  2022  5470 0.015358
              D: Nye Borgerlige   Copenhagen  2022  5400 0.015162
            O: Dansk Folkeparti   Copenhagen  2022  4862 0.013651
         K: Kristendemokraterne   Copenhagen  2022   757 0.002125
          

In [73]:
df.to_csv('data/denmark_election_results.csv', index=False)

# Turnout

In [74]:
turnout_api = DstApi('LABY09')

params = turnout_api.define_base_params(language='en')

df_t = election_api.get_data(params=params)
df_t.head()

,KOMGRP,VALRES,TID,INDHOLD
0,All Denmark,Personal voting percentage,2007,50.78
1,Capital municipalities,Advance vote percentage,2007,6.16
2,Copenhagen,Blanks voting percentage,2007,0.41
3,Copenhagen,Invalid vote percentage,2007,0.72
4,Frederiksberg,Voting percentage,2007,87.78


In [75]:
df_t.rename(columns={
    'VALRES': 'vote',
    'KOMGRP': 'municipality',
    'TID': 'year',
    'INDHOLD': 'N'
}, inplace=True)

# Keep only 2019 and 2022
df_t = df_t[df_t['year'].isin([2019, 2022])]

In [76]:
municipalities = ['Copenhagen', 'Frederiksberg', 'Ballerup', 'Brøndby', 'Dragør',
       'Gentofte', 'Gladsaxe', 'Glostrup', 'Herlev', 'Albertslund',
       'Hvidovre', 'Høje-Taastrup', 'Lyngby-Taarbæk', 'Rødovre', 'Ishøj',
       'Tårnby', 'Vallensbæk', 'Furesø', 'Allerød', 'Fredensborg',
       'Helsingør', 'Hillerød', 'Hørsholm', 'Rudersdal', 'Egedal',
       'Frederikssund', 'Greve', 'Køge', 'Halsnæs', 'Roskilde', 'Solrød',
       'Gribskov', 'Odsherred', 'Holbæk', 'Faxe', 'Kalundborg',
       'Ringsted', 'Slagelse', 'Stevns', 'Sorø', 'Lejre', 'Lolland',
       'Næstved', 'Guldborgsund', 'Vordingborg', 'Bornholm', 'Middelfart',
       'Christiansø', 'Assens', 'Faaborg-Midtfyn', 'Kerteminde', 'Nyborg',
       'Odense', 'Svendborg', 'Nordfyns', 'Langeland', 'Ærø', 'Haderslev',
       'Billund', 'Sønderborg', 'Tønder', 'Esbjerg', 'Fanø', 'Varde',
       'Vejen', 'Aabenraa', 'Fredericia', 'Horsens', 'Kolding', 'Vejle',
       'Herning', 'Holstebro', 'Lemvig', 'Struer', 'Syddjurs',
       'Norddjurs', 'Favrskov', 'Odder', 'Randers', 'Silkeborg', 'Samsø',
       'Skanderborg', 'Aarhus', 'Ikast-Brande', 'Ringkøbing-Skjern',
       'Hedensted', 'Morsø', 'Skive', 'Thisted', 'Viborg', 'Brønderslev',
       'Frederikshavn', 'Vesthimmerlands', 'Læsø', 'Rebild',
       'Mariagerfjord', 'Jammerbugt', 'Aalborg', 'Hjørring']

df_t = df_t[df_t['municipality'].isin(municipalities)].copy()


In [77]:
# keep only "Voting percentage"
df_t = df_t[df_t['vote'] == 'Voting percentage'].copy()
df_t.head()

,municipality,vote,year,N
242,Fanø,Voting percentage,2019,88.50
247,Fredericia,Voting percentage,2019,82.14
252,Herning,Voting percentage,2019,86.04
257,Syddjurs,Voting percentage,2019,84.59
262,Randers,Voting percentage,2019,82.09


In [78]:
df_t.to_csv('data/denmark_voter_turnout.csv', index=False)

In [79]:
# Merge turnout into df, scale party pcts by turnout, and add non-voter row
turnout = df_t[['municipality', 'year', 'N']].rename(columns={'N': 'turnout_pct'})
turnout['turnout_pct'] = turnout['turnout_pct'].apply(lambda x: pd.to_numeric(x, errors='coerce'))  # ensure numeric
turnout['turnout_pct'] = turnout['turnout_pct'] / 100  # convert to proportion

df = pd.read_csv('data/denmark_election_results.csv')
df_all = df.merge(turnout, on=['municipality', 'year'], how='inner')
df_all['pop_pct'] = df_all['N_pct'] * df_all['turnout_pct']

# Build non-voter rows: 1 - turnout for each municipality/year
nonvoters = turnout.copy()
nonvoters['party'] = 'Non-voter'
nonvoters['pop_pct'] = 1 - nonvoters['turnout_pct']

# Combine
df_combined = pd.concat([
    df_all[['party', 'municipality', 'year', 'pop_pct']],
    nonvoters[['party', 'municipality', 'year', 'pop_pct']]
], ignore_index=True)

df_combined.head(10)

,party,municipality,year,pop_pct
0,Independent candidate,Copenhagen,2019,0.001208
1,Independent candidate,Frederiksberg,2019,0.000947
2,Independent candidate,Ballerup,2019,0.002409
3,Independent candidate,Brøndby,2019,0.002994
4,Independent candidate,Dragør,2019,0.000490
5,Independent candidate,Gentofte,2019,0.002106
6,Independent candidate,Gladsaxe,2019,0.002021
7,Independent candidate,Glostrup,2019,0.002471
8,Independent candidate,Herlev,2019,0.002970
9,Independent candidate,Albertslund,2019,0.002423


In [81]:
sample = df_combined[(df_combined['municipality'] == 'Copenhagen') & (df_combined['year'] == 2022)].sort_values('pop_pct', ascending=False)
print(sample.to_string(index=False))

                          party municipality  year  pop_pct
                      Non-voter   Copenhagen  2022 0.166400
           A: Socialdemokratiet   Copenhagen  2022 0.147549
                Ø: Enhedslisten   Copenhagen  2022 0.126962
F: SF - Socialistisk Folkeparti   Copenhagen  2022 0.100084
                Å: Alternativet   Copenhagen  2022 0.086310
                 M: Moderaterne   Copenhagen  2022 0.076480
            I: Liberal Alliance   Copenhagen  2022 0.069332
        B: Det Radikale Venstre   Copenhagen  2022 0.065323
                     V: Venstre   Copenhagen  2022 0.065222
 C: Det Konservative Folkeparti   Copenhagen  2022 0.035512
                 Q: Frie Grønne   Copenhagen  2022 0.020718
        Æ: Danmarksdemokraterne   Copenhagen  2022 0.012802
              D: Nye Borgerlige   Copenhagen  2022 0.012639
            O: Dansk Folkeparti   Copenhagen  2022 0.011379
         K: Kristendemokraterne   Copenhagen  2022 0.001772
          Independent candidate   Copenh

In [83]:
df_combined.to_csv('data/denmark_election_results_with_turnout.csv', index=False)